# Migrate to Apache Iceberg
## Selective In-Place Migration for Large Historical Tables using the add_files procedure

### Configure Spark Session

In [ ]:
%%configure -f
{   
    "conf":{
        "numExecutors": 1, 
         "executorMemory": "14G",
         "executorCores": 4, 
        "spark.dynamicAllocation.enabled": "false",
        "spark.sql.extensions":"org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
         "spark.sql.catalog.spark_catalog":"org.apache.iceberg.spark.SparkSessionCatalog",
         "spark.sql.catalog.spark_catalog.type":"glue"
    }
}

### Configure variables

In [ ]:
source_table_name = "events_historical_hive"
target_table_name="events_historical_hive_iceberg"
bucket_name="mybucket"
db_name="mydb"

### Create db

In [ ]:
spark.sql(f"""
CREATE DATABASE IF NOT EXISTS {db_name}
LOCATION 's3://{bucket_name}/{db_name}'
""")

### Create and populate source table

In [ ]:
spark.sql(f"""
CREATE TABLE {db_name}.{source_table_name} (
    event_id STRING,
    user_id STRING,
    event_type STRING,
    event_timestamp TIMESTAMP,
    additional_data STRING,
    year INT,
    month INT
)
USING PARQUET
PARTITIONED BY (year,month)
LOCATION 's3://{bucket_name}/{db_name}/{source_table_name}/'
""")

In [ ]:
#To allow automatic creation of partitions when inserting data
spark.conf.set("hive.exec.dynamic.partition.mode", "nonstrict")

In [ ]:
#Generate sample data for 11 years (1 record per month)
from datetime import datetime

start_year = 2015
end_year   = 2025

values = []

for y in range(start_year, end_year + 1):
    for m in range(1, 13):
        event_id = f"EVT_{y}{m:02d}"
        user_id = f"USER_{m:03d}"
        event_type = "auto_event"
        timestamp_val = f"{y}-{m:02d}-01 12:00:00" 
        add_data = f"'data_{y}_{m}'"
        
        values.append(
            f"('{event_id}','{user_id}','{event_type}',timestamp('{timestamp_val}'),{add_data},{y},{m})"
        )

values_str = ",".join(values)


spark.sql(f"""
INSERT INTO {db_name}.{source_table_name} VALUES
{values_str}
""")

Check table partitions

In [ ]:
spark.sql(f"""
SHOW PARTITIONS {db_name}.{source_table_name}
""").show(20,truncate=False)

We have 11 years, 12 months per year so 132 months of data

In [ ]:
spark.sql(f"""
SELECT COUNT(DISTINCT CONCAT(year, '-', month)) as num_months
FROM {db_name}.{source_table_name}
""").show(truncate=False)

Verify Hive table

In [ ]:
spark.sql(f"""
SELECT * FROM {db_name}.{source_table_name} ORDER BY event_id LIMIT 10
""").show(truncate=False)

## In-Place migration of last 5 years of data

Create Iceberg table with same schema and partitioning of source table

In [ ]:
spark.sql(f"""
CREATE TABLE {db_name}.{target_table_name} (
    event_id STRING,
    user_id STRING,
    event_type STRING,
    event_timestamp TIMESTAMP,
    additional_data STRING,
    year INT,
    month INT
)
USING iceberg
PARTITIONED BY (year,month)
LOCATION 's3://{bucket_name}/{db_name}/{target_table_name}/'
""")

Let's do in-place migration of last 5 years of data

In [ ]:
for y in range(2021, 2026):
    spark.sql(f""" 
    CALL system.add_files( 
    source_table => '{db_name}.{source_table_name}',  
    table => '{db_name}.{target_table_name}',
    partition_filter => map('year', {y})
    )
    """).show(truncate=False)

In the Iceberg table we have 5 years, 12 months per year so 60 months of data

In [ ]:
spark.sql(f"""
SELECT COUNT(DISTINCT CONCAT(year, '-', month)) as num_months
FROM {db_name}.{target_table_name}
""").show(truncate=False)

Verify Iceberg table

In [ ]:
spark.sql(f"""
SELECT * FROM {db_name}.{target_table_name} ORDER BY event_id
""").show(truncate=False)

We can see the table is pointing to the source table data

In [ ]:
spark.sql(f"""
SELECT file_path,partition
FROM {db_name}.{target_table_name}.files
""").show(truncate=False)

**WARNING**: running Iceberg operations that phisically remove data files (eg., expire snapshots) potentially impact the source Hive table. One possibility could be to remove all partitions for years 2015-2025 from the source Hive table. Then consider the Iceberg table for the new data and the Hive table for the historical data.

Example to remove all partitions for years 2015-2025 from the source Hive table:

In [ ]:
for y in range(2021, 2026):
    spark.sql(f"""
    ALTER TABLE {db_name}.{source_table_name} DROP PARTITION (year={y})
    """).show(truncate=False)

We can see now the Hive table holds data from 2015 till 2020, 6 years so 72 months

In [ ]:
spark.sql(f"""
SELECT COUNT(DISTINCT CONCAT(year, '-', month)) as num_months
FROM {db_name}.{source_table_name}
""").show(truncate=False)